In [5]:
import numpy as np
import os
import subprocess
import time
import optuna
import matplotlib.pyplot as plt

In [ ]:
def wait_for_file(file_path, timeout=600, check_interval=5):

    start_time = time.time()
    while not os.path.exists(file_path):
        if time.time() - start_time > timeout:
            raise TimeoutError(f"File {file_path} was not created within {timeout} seconds.")
        time.sleep(check_interval)

def wait_for_file_creation(file_path, check_interval=10):
    
    print(f"Waiting for file: {file_path}")
    while not os.path.exists(file_path):
        time.sleep(check_interval)
    print(f"File found: {file_path}")

def target_function(x: np.array) -> np.array:

    target = x**3

    return target

def pearson_correlation_coefficient(x: np.array, y: np.array) -> float:

    N = x.shape[0]

    x_bar = np.mean(x)
    y_bar = np.mean(y)
    std_x = np.std(x, ddof=1)
    std_y = np.std(y, ddof=1)

    x_c = (x - x_bar) / std_x
    y_c = (y - y_bar) / std_y

    pcc = np.sum(x_c*y_c) / (N - 1)

    return pcc   

def objective(trial):

    print(f"[Trial {trial.number}] Starting trial...")

    V_MIN = -1.5
    V_MAX = 1.5

    control_indices = [1, 2, 3, 4, 6, 7]
    input_index = 5
    output_index = 0
    num_of_points = 100
    eq_steps = 10_000
    sim_steps = 10_000

    params = {
        idx: trial.suggest_float(f"c_{idx}", V_MIN, V_MAX)
        for idx in control_indices
    }

    function_args = [
        f"--file_name={trial.number}",
        f"--input_idx={input_index}",
        f"--output_idx={output_index}",
        f"--vMin={V_MIN}",
        f"--vMax={V_MAX}",
        f"--numOfPoints={num_of_points}",
        f"--equilibriumSteps={eq_steps}",
        f"--simulationSteps={sim_steps}",
        f"--configs=={}"
        f"--saveFolderPath={'../trials'}"
    ]

    control_voltage_args = []

    for idx, val in params.items():
        control_voltage_args.append(f"--c_v={idx}={val}")
    current_dir = os.getcwd()
    
    slurm_script_path = [os.path.abspath(os.path.join(current_dir, os.pardir, "slurm"))]
    slurm_script = os.path.join(slurm_script_path[0], "helix_optuna.sh")
    slurm_cmd = ["sbatch", slurm_script]
    #print(slurm_cmd)
    #print(trial.number)

    cmd = slurm_cmd + function_args + control_voltage_args
    subprocess.run(cmd, capture_output=True, text=True)

    npz_path = os.path.abspath(os.path.join(current_dir, os.pardir, os.pardir, "trials"))#f"../../trials/data_point{trial.number}.npz"
    npz_file = os.path.abspath(os.path.join(npz_path, f"data_point{trial.number}.npz"))
    #print(npz_path)
    #print(npz_file)
    wait_for_file_creation(npz_file, 10)

    file_name = npz_path
    data = np.load(file=file_name)
    curve = data["outputCurrent"]

    score = pearson_correlation_coefficient(curve)
    print(f"[{trial.number}] Finished with score: {score}")

    return score

In [7]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.MedianPruner() 
)

[I 2025-06-26 11:47:06,169] A new study created in memory with name: no-name-624a8025-199c-4427-bb7e-ab6ebf78b32c


In [8]:
study.optimize(
    objective,
    n_trials=10,
    timeout=None,
    gc_after_trial=True,
)

[Trial 0] Starting trial...
Waiting for file: /gpfs/bwfor/home/hd/hd_hd/hd_gy283/kmc_project/trials/data_point0.npz


[W 2025-06-26 11:53:52,736] Trial 0 failed with parameters: {'c_1': 1.1455243227105587, 'c_2': -0.8154350958003753, 'c_3': 0.18362431444185123, 'c_4': -1.092033860657481, 'c_6': 0.5236909666343723, 'c_7': 1.236855293974073} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/hd/hd_hd/hd_gy283/.conda/envs/py311/lib/python3.11/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/scratch/hd_gy283_o05i15/ipykernel_3903226/230595209.py", line 88, in objective
    wait_for_file_creation(npz_file, 10)
  File "/scratch/hd_gy283_o05i15/ipykernel_3903226/230595209.py", line 13, in wait_for_file_creation
    time.sleep(check_interval)
KeyboardInterrupt
[W 2025-06-26 11:53:52,739] Trial 0 failed with value None.


KeyboardInterrupt: 